# Aggregation Robustness Study

Session 3 showed that naive FedAvg aggregation reduced performance from approximately 75–79% to 47.5%.

Session 4 demonstrated that shared initialization improved aggregated model accuracy to 76.25%.

This experiment investigates whether the improvement is consistent across multiple random shared initializations.

The goal is to determine whether shared initialization reliably improves quantum parameter aggregation.

In [1]:
import numpy as np

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.utils.loss_functions import CrossEntropyLoss

from qiskit_algorithms.optimizers import COBYLA

In [2]:
# --------------------------------------------------
# Dataset
# --------------------------------------------------

X, y = make_moons(
    n_samples=400,
    noise=0.15,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# --------------------------------------------------
# Federated Split
# --------------------------------------------------

client1_idx = np.where(y_train == 0)[0][:128]
client1_idx = np.concatenate([
    client1_idx,
    np.where(y_train == 1)[0][:32]
])

client2_idx = np.where(y_train == 0)[0][128:158]
client2_idx = np.concatenate([
    client2_idx,
    np.where(y_train == 1)[0][32:160]
])

X_client1 = X_train[client1_idx]
y_client1 = y_train[client1_idx]

X_client2 = X_train[client2_idx]
y_client2 = y_train[client2_idx]

y_client1_q = 2 * y_client1 - 1
y_client2_q = 2 * y_client2 - 1
y_test_q = 2 * y_test - 1

In [3]:
# --------------------------------------------------
# Quantum Circuit
# --------------------------------------------------

num_qubits = 4

x_params = ParameterVector("x", 2)
theta = ParameterVector("θ", 8)

qc = QuantumCircuit(num_qubits)

qc.ry(x_params[0], 0)
qc.ry(x_params[1], 1)

for i in range(num_qubits):
    qc.ry(theta[2*i], i)
    qc.rz(theta[2*i + 1], i)

qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(2, 3)

observable = SparsePauliOp.from_list([
    ("ZZZZ", 1)
])

qnn = EstimatorQNN(
    circuit=qc,
    input_params=x_params,
    weight_params=theta,
    observables=observable
)

print("QNN Ready")

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


QNN Ready


In [4]:
def evaluate_fedavg(weights):

    predictions = []

    for sample in X_test:

        output = qnn.forward(
            input_data=sample,
            weights=weights
        )

        pred = 1 if output[0][0] >= 0 else -1
        predictions.append(pred)

    predictions = np.array(predictions)

    return np.mean(predictions == y_test_q)

### Main Experiment

In [5]:
results = []

for run in range(5):

    print(f"\n{'='*50}")
    print(f"RUN {run+1}")
    print(f"{'='*50}")

    shared_initial_weights = np.random.rand(8)

    client1 = NeuralNetworkClassifier(
        neural_network=qnn,
        optimizer=COBYLA(maxiter=100),
        loss=CrossEntropyLoss(),
        one_hot=False,
        initial_point=shared_initial_weights
    )

    client2 = NeuralNetworkClassifier(
        neural_network=qnn,
        optimizer=COBYLA(maxiter=100),
        loss=CrossEntropyLoss(),
        one_hot=False,
        initial_point=shared_initial_weights
    )

    client1.fit(X_client1, y_client1_q)
    client2.fit(X_client2, y_client2_q)

    global_weights = (
        client1.weights +
        client2.weights
    ) / 2

    fedavg_acc = evaluate_fedavg(global_weights)

    print(f"FedAvg Accuracy: {fedavg_acc:.4f}")

    results.append(fedavg_acc)


RUN 1
FedAvg Accuracy: 0.7625

RUN 2
FedAvg Accuracy: 0.7625

RUN 3
FedAvg Accuracy: 0.7625

RUN 4
FedAvg Accuracy: 0.7625

RUN 5
FedAvg Accuracy: 0.8000


In [6]:
print("\n")
print("="*60)
print("SUMMARY")
print("="*60)

for i, acc in enumerate(results):
    print(f"Run {i+1}: {acc:.4f}")

print("\nAverage Accuracy :", np.mean(results))
print("Std Deviation    :", np.std(results))
print("Best Run         :", np.max(results))
print("Worst Run        :", np.min(results))



SUMMARY
Run 1: 0.7625
Run 2: 0.7625
Run 3: 0.7625
Run 4: 0.7625
Run 5: 0.8000

Average Accuracy : 0.7699999999999999
Std Deviation    : 0.015000000000000036
Best Run         : 0.8
Worst Run        : 0.7625
